<a href="https://colab.research.google.com/github/gayakarapetyan/PythonCourseH2/blob/main/Session2/session2_codes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌧️ Session 2 — Precipitation & Discharge Analysis: Selke Catchment
### *Statistics & Machine Learning*

---

## Info

The **Selke** is a small river in Saxony-Anhalt, running from the Harz mountains
down to the lowland plain where it meets the Bode.

We have two datasets:
- **Precipitation** — 13 rain gauges across the catchment (daily, mm)
- **Discharge** — 3 river gauges along the Selke (daily, m³/s)

The big question connecting everything today:

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── File paths — change these to your Google Drive paths ─────────
PRECIP_PATH   = '/content/drive/MyDrive/Colab Notebooks/PythonCourse/Session2/Data/Stations_Selke_Precipitation.xlsx'
DISCHARGE_PATH = '/content/drive/MyDrive/Colab Notebooks/PythonCourse/Session2/Data/Discharge_Selke.xlsx'


---
## Part 1 — Loading the Data

### 1.1 Precipitation

13 stations, one Excel sheet each.
Columns: `Datum` (date) + `Daily_P` (mm).
Value `-999` = missing data — we replace it with `NaN`.


In [ ]:
# Station metadata, coordinates from DWD website
STATIONS = {
    'AU':  {'name': 'Arnstein-Ulzigerode',       'lat': 51.6794, 'lon': 11.3641, 'elev': 217},
    'AM':  {'name': 'Aschersleben-Mehringen',     'lat': 51.7259, 'lon': 11.5110, 'elev': 108},
    'BH':  {'name': 'Börde-Hakel-Hakeborn',       'lat': 51.9146, 'lon': 11.3615, 'elev': 116},
    'HG':  {'name': 'Harzgerode',                 'lat': 51.6520, 'lon': 11.1366, 'elev': 404},
    'HAT': {'name': 'Harztor-Ilfeld-Hufhaus',     'lat': 51.5985, 'lon': 10.8504, 'elev': 528},
    'HL':  {'name': 'Hecklingen-Groß Börnecke',   'lat': 51.8806, 'lon': 11.4650, 'elev': 104},
    'OH':  {'name': 'Oberharz am Brocken-Stiege', 'lat': 51.6647, 'lon': 10.8810, 'elev': 505},
    'QB':  {'name': 'Quedlinburg',                'lat': 51.7953, 'lon': 11.1320, 'elev': 142},
    'SN':  {'name': 'Seeland-Nachterstedt',       'lat': 51.8154, 'lon': 11.3242, 'elev': 123},
    'SA':  {'name': 'Selke-Aue-Hausneindorf',     'lat': 51.8426, 'lon': 11.2742, 'elev': 116},
    'ST':  {'name': 'Staßfurt',                   'lat': 51.8620, 'lon': 11.5514, 'elev':  67},
    'SH':  {'name': 'Südharz-Hayn/Harz',          'lat': 51.5683, 'lon': 11.0815, 'elev': 434},
    'WN':  {'name': 'Winningen',                  'lat': 51.8234, 'lon': 11.4558, 'elev': 142},
}

In [ ]:
for code in STATIONS:
  print(code)

In [ ]:
def load_precip(filepath, sheet_code):
    df = pd.read_excel(filepath, sheet_name=sheet_code, header=2)
    df.columns = ['Datum', 'Daily_P']
    df['Datum']   = pd.to_datetime(df['Datum'])
    df = df.set_index('Datum')
    df['Daily_P']  = df['Daily_P'].replace(-999, np.nan)
    return df

In [ ]:
# you can read data from one station:
#load_precip(PRECIP_PATH, 'WN')

In [ ]:
# Task 1: Load all stations into a dictionary using for loop
#your code here .....

In [ ]:
'''
# This code needs fixing !!!
precip = {}
for code in STATIONS:
df = load_precip(PRECIP_PATH, code)
precip[code] = df
print(f'\n{len(precip)} stations loaded.')
'''

### 1.2 Discharge

3 stations on the Selke — upstream to downstream:
- **Silberhütte** — upper catchment
- **Hausneindorf** — lower catchment

Columns: `Date, Jahr, Monat, Tag, Stunde, Abfluss_H`
`Stunde` is always 24 → one row per day.
Units: **m³/s** (cubic metres per second).


In [ ]:
DISCHARGE_STATIONS = ['Hausneindorf', 'Meisdorf', 'Silberhuette']

def load_discharge(filepath, sheet_name):
    df = pd.read_excel(filepath, sheet_name=sheet_name)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.set_index('Date')

    abfluss_col = [c for c in df.columns if str(c).startswith('Abfluss')][0]
    df = df[[abfluss_col]].rename(columns={abfluss_col: 'discharge'})
    df['discharge'] = pd.to_numeric(df['discharge'], errors='coerce')
    return df

In [ ]:
discharge = {}

for name in DISCHARGE_STATIONS:
    df = load_discharge(DISCHARGE_PATH, name)
    print(df.head())
    discharge[name] = df

---
## Part 2 — Precipitation: Single Station Analysis

We use **Harzgerode (HG)** as our main station — longest record, in the mountains.


### 2.1 — First look

In [ ]:
# Always inspect the data before calculating
df = precip['HG']
print(df.describe().round(2))
print(f'\nMissing: {df["Daily_P"].isna().sum():,} days')

### 2.2 — Histogram of daily rainfall

In [ ]:
# Distribution of rainy days
# We use only days with rain > 0 so the zero spike doesn't dominate
rainy = precip['HG'][precip['HG']['Daily_P'] > 0]['Daily_P'].dropna()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(rainy, bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_title('Daily precipitation distribution — Harzgerode (rainy days only)', fontsize=13)
ax.set_xlabel('mm / day')
ax.set_ylabel('Number of days')

# Mark mean and median
ax.axvline(rainy.mean(),   color='tomato',  linewidth=2, linestyle='-',
           label=f'Mean   {rainy.mean():.1f} mm')
ax.axvline(rainy.median(), color='orange', linewidth=2, linestyle='--',
           label=f'Median {rainy.median():.1f} mm')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# print statistics
print(f'Mean:   {rainy.mean():.2f} mm')
print(f'Median: {rainy.median():.2f} mm')
print(f'90th percentile: {np.percentile(rainy, 90):.1f} mm')
print(f'99th percentile: {np.percentile(rainy, 99):.1f} mm')

In [ ]:
# lets try resample() function
# lets prcatice on random data first
dates = pd.date_range(start = '2026-04-01', periods = 90, freq = 'D')
df = pd.DataFrame({'values':[10] * 90}, index = dates)

print(df.resample('ME').sum()) # ME- Month End, MS - Month Start, D- day, YE for year
monthly_totals = df.resample('ME').sum()

In [ ]:
# try groupby function
data = {
    'location': ['North', 'South','North', 'South','North', 'South'],
    'Station': ['Station1', 'Station2', 'Station3','Station4','Station5','Station6'],
    'precip': [10,8,15,12,20,18],
    'temp': [18,20,24,22,22,20]
}
df = pd.DataFrame(data)

In [ ]:
# groupby
basic_gb = df.groupby('location')['precip'].sum()
print(basic_gb)

### 2.3 — Monthly averages

In [ ]:
# TASK — Monthly average precipitation
# Resample to monthly totals, then group by month number and average across years
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

df = precip['HG']
monthly_totals = df['Daily_P'].resample('ME').sum()
avg_by_month   = monthly_totals.groupby(monthly_totals.index.month).mean()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(1,13), avg_by_month.values,
              color='steelblue', edgecolor='white')
bars[avg_by_month.values.argmax()].set_color('tomato')

ax.set_xticks(range(1,13))
ax.set_xticklabels(MONTH_NAMES)
ax.set_title('Average monthly precipitation — Harzgerode', fontsize=13)
ax.set_ylabel('Average total (mm)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f'Wettest month: {MONTH_NAMES[avg_by_month.idxmax()-1]}  ({avg_by_month.max():.0f} mm)')
print(f'Driest month:  {MONTH_NAMES[avg_by_month.idxmin()-1]}  ({avg_by_month.min():.0f} mm)')


### 2.4 — Annual totals + trend

In [ ]:
#TASK — Annual totals
#hints  .resample('YE').sum()

### 2.5 — Rain classifier function

In [ ]:
# TASK — Classify each day using if/elif/else
# you can use keys as 'light', 'heavy', 'moderate'

def classify_rain(mm):
  # write the content here....

# .apply() calls the function once for every row — like a loop, but faster
#df = precip['HG'].copy()
#df['category'] = df['Daily_P'].apply(classify_rain)

---
## Part 3 — Precipitation: All Stations Together


### 3.1 — Annual totals comparison

In [ ]:
#One dataframe with all stations as columns
annual_df = pd.DataFrame({
    code: precip[code]['Daily_P'].resample('YE').sum()
    for code in precip
})

# Mountain stations (high elevation) vs plain stations
mountain = ['OH','HAT','SH','HG']

fig, ax = plt.subplots(figsize=(13, 5))
for col in annual_df.columns:
    lw    = 2.2 if col in mountain else 0.9
    alpha = 0.9 if col in mountain else 0.4
    ax.plot(annual_df.index.year, annual_df[col],
            label=col, linewidth=lw, alpha=alpha)

ax.set_title('Annual precipitation — all 13 stations\n(bold = mountain stations)',
             fontsize=13)
ax.set_ylabel('mm / year')
ax.legend(ncol=5, fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print('Long-term mean annual precipitation:')
lt = annual_df.mean().sort_values(ascending=False)
for code, val in lt.items():
    print(f'  {code}  {STATIONS[code]["name"]:<32} {val:.0f} mm  (elev {STATIONS[code]["elev"]} m)')

### 3.2 — Correlation between stations

In [ ]:
# TASK — Correlation matrix
# On a given day, if it rains at HG, does it also rain at OH? At ST?
daily_df = pd.DataFrame({code: precip[code]['Daily_P'] for code in precip})
corr     = daily_df.corr()

fig, ax = plt.subplots(figsize=(10, 9))
mat = ax.matshow(corr.values, cmap='RdYlBu_r', vmin=0.3, vmax=1.0)
plt.colorbar(mat, ax=ax, shrink=0.8, label='Pearson r')

codes = list(corr.columns)
ax.set_xticks(range(len(codes))); ax.set_xticklabels(codes, rotation=45, ha='left')
ax.set_yticks(range(len(codes))); ax.set_yticklabels(codes)

for i in range(len(codes)):
    for j in range(len(codes)):
        ax.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center',
                fontsize=7, color='white' if corr.values[i,j] > 0.75 else 'black')

ax.set_title('Daily precipitation correlation — all stations', fontsize=13, pad=20)
plt.tight_layout()
plt.show()

corr_no_diag = corr.copy()
np.fill_diagonal(corr_no_diag.values, np.nan)
max_pair = corr_no_diag.stack().idxmax()
min_pair = corr_no_diag.stack().idxmin()

---
## Part 4 — Discharge: Loading & Inspection

Now we bring in the river itself.
Three gauges on the Selke — same river, different positions.


In [ ]:
# Plot all 3 discharge stations together
fig, ax = plt.subplots(figsize=(13, 4))

colors = {'Silberhuette': 'steelblue',
          'Meisdorf':     'tomato',
          'Hausneindorf': 'seagreen'}

for name, df in discharge.items():
    ax.plot(df.index, df['discharge'],
            label=name.capitalize(), color=colors[name],
            linewidth=0.8, alpha=0.85)

ax.set_title('Daily discharge — all 3 Selke gauges', fontsize=13)
ax.set_ylabel('Discharge (m³/s)')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## Part 5 — Discharge: Seasonal Patterns


In [ ]:
# Monthly average discharge for all 3 stations
# Same approach as precipitation monthly averages
# hints! you can use .resample('ME'), .groupby(monthly.index.month)
#Write your code here...............



In [ ]:
# Annual maximum discharge per year (flood peaks)
# For each year, what was the highest single-day discharge?
# hints!  .resample('YE').max(), you can also use .index.year to get each year


---
## Part 6 — Combining Precipitation and Discharge

This is the heart of hydrology.
Rain falls on the catchment → some becomes river flow → but when?


### 6.1 — Catchment average precipitation

In [ ]:
# Spatial average: mean of all 13 stations = catchment rainfall

daily_df  = pd.DataFrame({code: precip[code]['Daily_P'] for code in precip})
catchment_rain = daily_df.mean(axis=1)   # mean across all station columns
catchment_rain.name = 'catchment_P'

print(f'Catchment average: {catchment_rain.mean():.2f} mm/day')
print(f'Max single day:    {catchment_rain.max():.1f} mm')


### 6.2 — The hydrograph plot

In [ ]:
# Plot precipitation and discharge
# Choose a year that is interesting to you or better find a day of flood

year = 2002
rain = catchment_rain[catchment_rain.index.year == year]
q    = discharge['Hausneindorf'][discharge['Hausneindorf'].index.year == year]

fig = plt.figure(figsize=(14, 6))
gs  = gridspec.GridSpec(2, 1, height_ratios=[1, 2], hspace=0.05)

# Top panel — precipitation (inverted bars)
ax1 = fig.add_subplot(gs[0])
ax1.bar(rain.index, rain.values, color='steelblue', width=1, alpha=0.8)
ax1.invert_yaxis()   # ← inverted! Rain bars grow downward — standard convention
ax1.set_ylabel('Precip (mm)', fontsize=10)
ax1.set_title(f'Catchment precipitation & Hausneindorf discharge — {year}', fontsize=13)
ax1.set_xlim(rain.index[0], rain.index[-1])
ax1.tick_params(labelbottom=False)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Bottom panel — discharge
ax2 = fig.add_subplot(gs[1])
ax2.plot(q.index, q['discharge'], color='seagreen', linewidth=1.8)
ax2.fill_between(q.index, q['discharge'], alpha=0.25, color='seagreen')
ax2.set_ylabel('Discharge (m³/s)', fontsize=10)
ax2.set_xlim(rain.index[0], rain.index[-1])
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### 6.3 — Lag time calculation

In [ ]:
# TASK — Measure the lag time between rain peak and discharge peak
# We look at one specific flood event and find both peaks manually

# Merge rain and discharge into one dataframe
merged = pd.merge(
    catchment_rain.rename('rain'),
    discharge['Hausneindorf']['discharge'],
    left_index=True, right_index=True, how='inner'
)

# Filter to a specific flood month — August 2002
event = merged['2002-08-01':'2002-08-31']

# Find peak dates
rain_peak_date = event['rain'].idxmax()
q_peak_date    = event['discharge'].idxmax()
lag_days       = (q_peak_date - rain_peak_date).days

print(f'Peak rainfall  : {event["rain"].max():.1f} mm  on {rain_peak_date.date()}')
print(f'Peak discharge : {event["discharge"].max():.1f} m³/s  on {q_peak_date.date()}')
print(f'Lag time       : {lag_days} days')

# Plot the event
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.bar(event.index, event['rain'], color='steelblue', width=1)
ax1.axvline(rain_peak_date, color='navy', linestyle='--', linewidth=1.5,
            label=f'Rain peak: {rain_peak_date.date()}')
ax1.set_ylabel('Catchment rain (mm)')
ax1.legend(fontsize=9)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

ax2.plot(event.index, event['discharge'], color='seagreen', linewidth=2)
ax2.axvline(q_peak_date, color='darkgreen', linestyle='--', linewidth=1.5,
            label=f'Q peak: {q_peak_date.date()}  (lag: {lag_days} d)')
ax2.set_ylabel('Discharge (m³/s)')
ax2.legend(fontsize=9)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

ax2.set_title(f'Flood event August 2002 — lag time = {lag_days} days', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Find an interesting days and plot both data, try to explain the results

### 6.4 — Upstream vs downstream response

In [ ]:
#  Testing all 3 gauges during the same flood peak
# Silberhütte is upstream , Hausneindorf is downstream

event_q = {name: discharge[name]['2002-08-01':'2002-08-31']
           for name in DISCHARGE_STATIONS}

fig, ax = plt.subplots(figsize=(12, 4))
for name, df in event_q.items():
    peak_date = df['discharge'].idxmax()
    ax.plot(df.index, df['discharge'],
            label=f'{name.capitalize()}  (peak: {peak_date.date()})',
            color=colors[name], linewidth=2)

ax.set_title('Flood propagation — August 2002: upstream to downstream', fontsize=13)
ax.set_ylabel('Discharge (m³/s)')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

### 6.5 — Runoff coefficient

In [ ]:
# What fraction of rain becomes river flow?
# Just an estimate
# Runoff coefficient C = total discharge volume / total rainfall volume
#
# Unit conversion:
#   Discharge is in m³/s  →  multiply by 86400 (seconds/day) → m³/day
#   Rainfall is in mm     →  multiply by catchment area in m²  → m³
#   Selke catchment area ≈ 468 km² = 468,000,000 m²

CATCHMENT_AREA_M2 = 468e6  # m²

common_start = max(catchment_rain.index.min(),
                   discharge['Hausneindorf'].index.min())
common_end   = min(catchment_rain.index.max(),
                   discharge['Hausneindorf'].index.max())

rain_period = catchment_rain[common_start:common_end].dropna()
q_period    = discharge['Hausneindorf']['discharge'][common_start:common_end].dropna()

# Total volumes
total_rain_m3 = rain_period.sum() / 1000 * CATCHMENT_AREA_M2
total_q_m3    = q_period.sum() * 86400

C = total_q_m3 / total_rain_m3

print(f'Total rain volume : {total_rain_m3/1e9:.2f} km³')
print(f'Total river flow  : {total_q_m3/1e9:.2f} km³')
print(f'Runoff coefficient: {C:.3f}  ({C*100:.1f}% of rain becomes river flow)')

---
## Part 7 — Correlation and Linear Regression



In [ ]:
# Build the merged dataframe
merged = pd.merge(
    catchment_rain.rename('rain'),
    discharge['Hausneindorf']['discharge'],
    left_index=True, right_index=True, how='inner'
).dropna()

# Shift discharge by 1 day — rain today, discharge tomorrow
merged['discharge_next'] = merged['discharge'].shift(-1)
merged = merged.dropna()

print(f'Total overlapping days: {len(merged):,}')
print()

# How many days have near-zero rain?
zero_rain  = (merged['rain'] == 0).sum()
small_rain = (merged['rain'] < 1).sum()
print(f'Days with exactly 0 mm rain : {zero_rain:,}  ({zero_rain/len(merged)*100:.0f}%)')
print(f'Days with < 1 mm rain       : {small_rain:,}  ({small_rain/len(merged)*100:.0f}%)')

In [ ]:
# The naive correlation

r_same = merged['rain'].corr(merged['discharge'])
r_next = merged['rain'].corr(merged['discharge_next'])

print('Naive correlation (all days including zeros):')
print(f'  rain vs discharge same day : r = {r_same:.3f}')
print(f'  rain vs discharge next day : r = {r_next:.3f}')
print()
print('Very low! Let us look at why with a scatter plot.')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# All days
ax = axes[0]
ax.scatter(merged['rain'], merged['discharge_next'],
           alpha=0.1, s=8, color='steelblue')
ax.set_xlabel('Catchment rain today (mm)')
ax.set_ylabel('Discharge tomorrow (m³/s)')
ax.set_title(f'All days  —  R²={r_next**2:.3f}', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Distribution of rain values
ax = axes[1]
ax.hist(merged['rain'], bins=60, color='steelblue',
        edgecolor='white', linewidth=0.3)
ax.set_xlabel('Catchment rain (mm)')
ax.set_ylabel('Number of days')
ax.set_title('Distribution of daily catchment rain', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Filter to meaningful rain events only
# We keep only days in the top 10% of rainfall
# Calculate the 90th percentile threshold
# Option A: top 10% of ALL days
threshold_all  = np.percentile(merged['rain'], 90)
# Option B: top 10% of RAINY days only (rain > 0)
# threshold_rainy = np.percentile(merged[merged['rain'] > 0]['rain'], 90)

print(f'90th percentile threshold (all days)  : {threshold_all:.1f} mm')
# print(f'90th percentile threshold (rainy days): {threshold_rainy:.1f} mm')
print()

# Filter
extreme = merged[merged['rain'] >= threshold_all].copy()
print(f'Days above threshold: {len(extreme):,}  ({len(extreme)/len(merged)*100:.1f}% of all days)')

In [ ]:
extreme.describe()

In [ ]:
# Calculate the 3-day cumulative rain before each discharge measurement.

merged['rain_3day'] = merged['rain'].rolling(window=3).sum()
merged['rain_5day'] = merged['rain'].rolling(window=5).sum()
merged = merged.dropna()

# Filter extreme events using 3-day cumulative rain
threshold_3day = np.percentile(merged['rain_3day'], 90)
extreme_3day   = merged[merged['rain_3day'] >= threshold_3day].copy()

print(f'3-day cumulative threshold (90th pct): {threshold_3day:.1f} mm')
print(f'Events above threshold: {len(extreme_3day):,}')

# Compare correlations
r_1day = extreme['rain'].corr(extreme['discharge_next'])
r_3day = extreme_3day['rain_3day'].corr(extreme_3day['discharge_next'])

print()
print('Correlation with discharge (extreme events only):')
print(f'  Single day rain    : r = {r_1day:.3f}  R² = {r_1day**2:.3f}')
print(f'  3-day cumul. rain  : r = {r_3day:.3f}  R² = {r_3day**2:.3f}')
print()
print('The 3-day cumulative rain is a better predictor — physically meaningful!')



In [ ]:
# Scatter plot and linear regression on clean data

x = extreme_3day['rain_3day'].values
y = extreme_3day['discharge_next'].values

slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
x_line = np.linspace(x.min(), x.max(), 200)
y_line = slope * x_line + intercept

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(x, y, alpha=0.3, s=15, color='steelblue',
           label='Extreme rain events (top 10%)')
ax.plot(x_line, y_line, color='tomato', linewidth=2.5,
        label=f'y = {slope:.3f}x + {intercept:.2f}\nR² = {r_value**2:.3f}  p = {p_value:.1e}')

ax.set_xlabel('3-day cumulative catchment rain (mm)', fontsize=11)
ax.set_ylabel('Discharge next day at Hausneindorf (m³/s)', fontsize=11)
ax.set_title('Rainfall → Discharge: extreme events, 3-day cumulative rain', fontsize=13)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f'Slope    : {slope:.3f}')
print(f'         → each extra mm of 3-day rain adds {slope:.3f} m³/s to next day discharge')
print(f'R²       : {r_value**2:.3f}')
print(f'p-value  : {p_value:.2e}  ({"statistically significant ✅" if p_value < 0.05 else "not significant ❌"})')
print()
print('Compared to naive approach:')
print(f'  Naive R²  (all days, 1-day rain)    : {r_next**2:.3f}')
print(f'  Better R² (top 10%, 3-day rain)     : {r_value**2:.3f}')




### 7.3 — Seasonal effect

In [ ]:
# ⚡ STRETCH — Step 6: does the relationship differ by season?
# Winter: soil saturated → more rain becomes runoff → steeper slope
# Summer: dry soil absorbs more → flatter slope

seasons = {
    'Winter (DJF)': ([12, 1, 2],  'steelblue'),
    'Spring (MAM)': ([3, 4, 5],   'seagreen'),
    'Summer (JJA)': ([6, 7, 8],   'tomato'),
    'Autumn (SON)': ([9, 10, 11], 'orange')
}

fig, ax = plt.subplots(figsize=(10, 6))

print(f'{"Season":<15} {"Slope":>8} {"R²":>8} {"N events":>10}')
print('-' * 45)

for season, (months, color) in seasons.items():
    mask = extreme_3day.index.month.isin(months)
    sub  = extreme_3day[mask]
    if len(sub) < 20:
        continue

    x_s = sub['rain_3day'].values
    y_s = sub['discharge_next'].values
    sl, ic, rv, pv, _ = stats.linregress(x_s, y_s)

    x_line = np.linspace(x_s.min(), x_s.max(), 100)
    ax.plot(x_line, sl * x_line + ic, linewidth=2.5, color=color,
            label=f'{season}  slope={sl:.3f}  R²={rv**2:.2f}')
    ax.scatter(x_s, y_s, alpha=0.15, s=10, color=color)

    print(f'{season:<15} {sl:>8.3f} {rv**2:>8.3f} {len(sub):>10}')

ax.set_xlabel('3-day cumulative catchment rain (mm)', fontsize=11)
ax.set_ylabel('Discharge next day (m³/s)', fontsize=11)
ax.set_title('Seasonal effect on rainfall-discharge relationship\n(extreme events only)', fontsize=13)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# 💬 Discussion:
# Which season has the steepest slope?
# Does it match your physical expectation?
# Why might autumn have a different pattern than spring
# even though temperatures are similar?


---
## Part 8 — 🌧️ Mini-Project: Annual Rain & Flood Report

Write `annual_report(year)` that combines both datasets and prints
a complete summary for any year.

Include:
1. Total catchment rainfall vs long-term average
2. Peak discharge at Hausneindorf — date and value
3. Number of heavy rain days (> 10 mm)
4. Longest dry spell
5. The hydrograph plot for that year

Test with 2002 (flood) and 2018 (drought).


In [ ]:
def annual_report(year):
    """Complete annual summary combining precipitation and discharge."""

    # ── Filter to year ────────────────────────────────────────────
    rain_y = catchment_rain[catchment_rain.index.year == year].fillna(0)
    q_y    = discharge['Hausneindorf'][
                discharge['Hausneindorf'].index.year == year]

    if len(rain_y) == 0:
        print(f'No data for {year}')
        return

    # ── Stats ─────────────────────────────────────────────────────
    total_rain   = rain_y.sum()
    lt_avg       = catchment_rain.resample('YE').sum().mean()
    lt_std       = catchment_rain.resample('YE').sum().std()
    z            = (total_rain - lt_avg) / lt_std
    verdict      = ('unusually WET 🌊' if z > 1 else
                    'unusually DRY 🔥' if z < -1 else
                    'within normal range ✅')

    heavy_days   = (rain_y > 10).sum()
    peak_q_val   = q_y['discharge'].max()
    peak_q_date  = q_y['discharge'].idxmax()
    dry_len, dry_start = find_dry_spell(precip['HG'][precip['HG'].index.year == year])

    # ── Print ─────────────────────────────────────────────────────
    print(f'╔══ Annual Report {year} ══╗')
    print(f'  Catchment rainfall  : {total_rain:.0f} mm  (avg {lt_avg:.0f} mm, {total_rain-lt_avg:+.0f} mm)')
    print(f'  Verdict             : {verdict}')
    print(f'  Heavy rain days     : {heavy_days}  (> 10 mm)')
    if dry_start:
        print(f'  Longest dry spell   : {dry_len} days from {dry_start.date()}')
    print(f'  Peak discharge      : {peak_q_val:.1f} m³/s  on {peak_q_date.date()}')
    print()

    # ── Plot ──────────────────────────────────────────────────────
    fig = plt.figure(figsize=(14, 6))
    gs  = gridspec.GridSpec(2, 1, height_ratios=[1, 2], hspace=0.05)

    ax1 = fig.add_subplot(gs[0])
    bar_colors = ['tomato' if v > 10 else 'steelblue' for v in rain_y.values]
    ax1.bar(rain_y.index, rain_y.values, color=bar_colors, width=1)
    ax1.invert_yaxis()
    ax1.set_ylabel('Rain (mm)')
    ax1.tick_params(labelbottom=False)
    orange_p = mpatches.Patch(color='tomato',   label='Heavy (>10 mm)')
    blue_p   = mpatches.Patch(color='steelblue', label='Normal')
    ax1.legend(handles=[blue_p, orange_p], loc='lower right', fontsize=9)
    ax1.set_title(f'Annual Report {year} — {verdict}', fontsize=13)
    ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

    ax2 = fig.add_subplot(gs[1])
    ax2.plot(q_y.index, q_y['discharge'], color='seagreen', linewidth=1.8)
    ax2.fill_between(q_y.index, q_y['discharge'], alpha=0.2, color='seagreen')
    ax2.axhline(q_y['discharge'].mean(), color='grey', linestyle='--',
                linewidth=1, label=f'Mean {q_y["discharge"].mean():.1f} m³/s')
    ax2.set_ylabel('Discharge (m³/s)')
    ax2.legend(fontsize=9)
    ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

annual_report(2002)
annual_report(2018)


### Run the report for your own chosen year
Pick any year from the dataset. What happened? Was it unusual?


In [ ]:
# Your code here
# annual_report(YOUR_YEAR)
